<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Régression linéaire <br>
RMSE public : <br>
RMSE perso : 2.1756
</div>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des bibliothèques
</div>

In [1]:
from Donnees_et_RMSE import *
from sklearn.linear_model import LinearRegression
import gc
from vacances_scolaires_france import SchoolHolidayDates
from jours_feries_france import JoursFeries

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Paramètres
</div>

In [2]:
# Durée maximale d'un trou sur laquelle on interpole linéairement, si c'est plus on utilise la prédiction du modèle
INTERPOLATION_MAX = pd.Timedelta(hours=6)

# Chevauchement maximal pour fusionner deux stations
CHEVAUCHEMENT_MAX = pd.Timedelta(hours=6)

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Fusion des stations renommées
</div>

In [3]:
def fusion_doublons(train_modele):
    print(f"Nombre de stations dans train : {len(train_modele.reset_index()["station"].unique())}")
    
    # Initialisation -------------------------------------------------------------------------------------------
    # On convertit les indexe en colonnes classiques
    stations = stations_toulouse.reset_index()
    train = train_modele.reset_index()
    
    # On extrait tous les indexe station et commit_at_index sous forme de liste
    stations_idx = train_modele.index.get_level_values("station")
    dates_idx = train_modele.index.get_level_values("commit_at")
    
    
    
    # Détection des doublons -----------------------------------------------------------------------------------
    # On détecte toutes les stations qui ont exactement les mêmes coordonnées
    doublons_coords = stations[stations.duplicated(subset=["longitude", "latitude"], keep=False)].dropna()
    doublons = doublons_coords.groupby(["longitude", "latitude"])["station"].apply(list)
    
    # On créé des datasets avec en index la staion et en colonne sa première / dernière date d'enregistrement
    debut = pd.Series(dates_idx, index=stations_idx).groupby(level="station").min()
    fin = pd.Series(dates_idx, index=stations_idx).groupby(level="station").max()
    
    # Par défaut, chaque station pointe vers elle-même
    nom_vers_ancien = {station: station for station in debut.index}
    
    # Puis on met à jour les doublons pour qu'ils pointent vers l'uinque station de référence (1er nom de la station)
    # Liste des groupes de stations renommées qui ne se chevauchent pas (ou presque)
    chaines = []
    for groupe in doublons:
        # On ignore les stations absentes de train et on trie du plus ancien au plus récent
        groupe = sorted([n for n in groupe if n in debut.index], key = lambda n: debut[n])
        # Groupe valide (pas de chevauchement) contstrui à partir de groupe
        chaine = []
        for nom in groupe:
            if chaine and fin[chaine[-1]] - debut[nom] > CHEVAUCHEMENT_MAX:
                chaines.append(chaine)
                chaine = []
            chaine.append(nom)
        if chaine:
            chaines.append(chaine)
    # Pour chaque groupe on prend le premier dans l'ordre chronologique comme référent
    for chaine in chaines:
        for nom in chaine:
            nom_vers_ancien[nom] = chaine[0]
    
    
    
    # Fusion des doublons --------------------------------------------------------------------------------------
    # On ne modifie que les stations renommées pour alléger le map
    renommees = {k: v for k, v in nom_vers_ancien.items() if k != v}
    stations_renommees = train["station"].isin(list(renommees))
    train.loc[stations_renommees, "station"] = train.loc[stations_renommees, "station"].map(renommees)
    
    # On supprime les doublons créés (si il y a du chevauchement de dates)
    stations_fusionnees = train["station"].isin(set(renommees.values()))
    doublons_dates = train.loc[stations_fusionnees].duplicated(subset=["station", "commit_at"], keep="last")
    train = train.drop(index=doublons_dates.index[doublons_dates])
    
    # On remet les indexes de train
    train = train.set_index(["station", "commit_at"]).sort_index()
    
    print(f"Nombre de stations UNIQUES dans train : {len(train.reset_index()["station"].unique())}")
    
    return train, nom_vers_ancien

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Interpolation linéaire
</div>

In [4]:
def interpoler_trous(train_modele):
    print(f"Nombre de mesures de base dans train : {len(train_modele.reset_index()["station"])}")
    
    # Initialisation -------------------------------------------------------------------------------------------
    # On convertit les indexe en colonnes classiques et on le trie pour être sûr que les date sont dans l'ordre croissant 
    train = train_modele.reset_index().sort_values(["station", "commit_at"])
    
    # Durée d'une mesure
    TMP = pd.Timedelta(minutes=15)
    
    # Détection des doublons -----------------------------------------------------------------------------------
    # Mesures juste avant le trou
    train_stations = train.groupby("station")
    train["debut_trou"] = train_stations["commit_at"].shift()
    train["debut_bikes"] = train_stations["bikes"].shift()
    train["debut_stands"] = train_stations["stands"].shift()
    # Ecart de temps entre les 2 mesures
    train["ecart"] = train["commit_at"] - train["debut_trou"]
    
    # On liste tous les trous inférieurs à INTERPOLATION_MAX et supérieurs à 15 min sinon ca ne serait pas un trou
    trous = train[(train["ecart"] > TMP) & (train["ecart"] <= INTERPOLATION_MAX)].copy()
    
    # Nombre de mesures manquantes dans chaque trou
    trous["manquantes"] = trous["ecart"] // TMP - 1
    
    if trous.empty:
        return train_modele
    
    # Interpolation linéaire -----------------------------------------------------------------------------------
    morceaux = []
    for trou in trous.itertuples():
        # On genère toutes les dates manquantes, en omettant la première et la dernière qu'on connait déja
        dates = pd.date_range(start=trou.debut_trou, end=trou.commit_at, freq=TMP)[1:-1]
        n = len(dates)
        morceaux.append(pd.DataFrame({
            "station": trou.station,
            "commit_at": dates,
            "bikes": np.linspace(trou.debut_bikes, trou.bikes, n+2)[1:-1],
            "stands": np.linspace(trou.debut_stands, trou.stands, n+2)[1:-1],
        }))
    
    # On rajoute tout d'un coup
    interpolations = pd.concat(morceaux)
    interpolations[["bikes", "stands"]] = interpolations[["bikes", "stands"]].round().astype(int)
    interpolations = interpolations.set_index(["station", "commit_at"])
    
    train_interpole = pd.concat([train_modele, interpolations]).sort_index()
    
    print(f"Nombre de mesures après interpolation dans train : {len(train_interpole.reset_index()["station"])}")
    
    return train_interpole

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Modèle de prévision
</div>

In [5]:
# train_modele sont les données avec lesquelles le modèle va s'entraîner
# train_test est le fichier dans lequel le modèle va récupérer les dates à prédire et dans lequel il va écrire ses prédictions
# nom_vers_ancien est la correspondance entre les noms dans test et les noms sur lesquels on a entraîné le modèle
def modele_prevision(train_modele, test_modele):
    # Initialisation -------------------------------------------------------------------------------------------
    # On fusione les données des stations qui ont été renommées (exactement les mêmes coordonnées)
    train_modele, nom_vers_ancien = fusion_doublons(train_modele)
    
    # On bouche les petits trous par interpolation linéaire
    train_modele = interpoler_trous(train_modele)
    
    # On convertit les indexe en colonnes classiques
    train = train_modele.reset_index()
    
    # On extrait tous les indexe station et commit_at_index sous forme de liste
    station_index = test_modele.index.get_level_values("station")
    commit_at_index = test_modele.index.get_level_values("commit_at")
    
    
    
    # Extraction des paramètres --------------------------------------------------------------------------------
    # Prise en compte de la date
    train["annee"] = train["commit_at"].dt.year
    train["mois"] = train["commit_at"].dt.month
    train["jour"] = train["commit_at"].dt.dayofweek
    train["weekend"] = (train["jour"] >= 5).astype(int)
    train["heure"] = train["commit_at"].dt.hour
    train["minute"] = train["commit_at"].dt.minute
    
    # On encode les variables temporelles de manière cyclique pour qu'il n'y ait pas de rupture (ex : 23h -> 0h)
    train["mois_sin"] = np.sin(2 * np.pi * train["mois"] / 12)
    train["mois_cos"] = np.cos(2 * np.pi * train["mois"] / 12)
    train["jour_sin"] = np.sin(2 * np.pi * train["jour"] / 7)
    train["jour_cos"] = np.cos(2 * np.pi * train["jour"] / 7)
    train["heure_sin"] = np.sin(2 * np.pi * train["heure"] / 24)
    train["heure_cos"] = np.cos(2 * np.pi * train["heure"] / 24)
    train["minute_sin"] = np.sin(2 * np.pi * train["minute"] / 60)
    train["minute_cos"] = np.cos(2 * np.pi * train["minute"] / 60)
    
    # Prise en compte de la géolocalisation
    train = train.merge(stations_toulouse[["station", "longitude", "latitude"]], on="station", how="left")
    
    # Prise en compte du nombre total de stations
    train["total"] = train["bikes"] + train["stands"]
    
    # Prise en compte du taux d'occupation de la station + on évite la division par 0
    train["occupation"] = train["bikes"] / train["total"].replace(0, 1)
    
    # Prise en compte des vacances scolaires (Toulouse est en zone C)
    # On extrait toutes les dates de train et test
    dates_train = train["commit_at"].dt.date.unique()
    dates_test = commit_at_index.date
    # On en extrait toutes les dates différentes
    toutes_dates = pd.Series(list(dates_train) + list(dates_test)).unique()
    # Pour chaque date on détecte si Toulouse est en vacance ou non
    d = SchoolHolidayDates()
    vacances_par_date = {date: d.is_holiday_for_zone(date, 'C') for date in toutes_dates}
    train["vacances"] = train["commit_at"].dt.date.map(vacances_par_date).astype(int)
    
    # Prise en compte des jours fériés
    # Toutes les années dans train union (|) test
    annees = set(train["annee"].unique()) | set(commit_at_index.year.unique())
    # On construit un set de tous les jours feriés de toutes les années détectées
    tous_feries = set()
    for annee in annees:
        tous_feries.update(JoursFeries.for_year(int(annee)).values())
    # On écrit le résultat dans train
    train["ferie"] = train["commit_at"].dt.date.isin(tous_feries).astype(int)
    
    # Données utiles pour l'entrainement du modèle
    colonnes_train = ["bikes",
                  "annee",
                  "mois_sin", "mois_cos",
                  "jour_sin", "jour_cos",
                  "weekend",
                  "heure_sin", "heure_cos",
                  "minute_sin", "minute_cos",
                  "longitude", "latitude",
                  "total", "occupation",
                 "vacances", "ferie"]
    
    # On liste les colonnes réellement utiles à l'entrainement
    colonnes_necessaires = list(dict.fromkeys(["station", "commit_at"] + colonnes_train))
    
    # On en déduit celles qui ne le sont pas
    colonnes_a_supprimer = [c for c in train.columns if c not in colonnes_necessaires]
    
    # On supprime les colonnes non-nécessaires (pour alléger le copy dans la boucle d'entrainement
    train = train.drop(columns=colonnes_a_supprimer)
    
    
    
    # Paramètres à l'instant initial ---------------------------------------------------------------------------
    # On récupère le nombre de vélos à l'instant zero du test
    instant_zero = list(zip(station_index, commit_at_index))
    bikes_train = history_toulouse_train["bikes"]
    stands_train = history_toulouse_train["stands"]
    
    # On localise chaque index (station, date) pour trouver le nombre de vélos et d'emplacement vides à l'instant zero du test
    bikes_zero = pd.Series(bikes_train.loc[instant_zero].values, index=test_modele.index)
    stands_zero = pd.Series(stands_train.loc[instant_zero].values, index=test_modele.index)
    
    # On récupère la géolocalisation de chaque station du test
    # On définit la colonne "station" comme étant l'indexe
    stations_pos = stations_toulouse.set_index("station").reindex(station_index)
    
    # On crée un dataframe avec toutes les données du test dont on aura besoin pour prédire.
    # Il faut qu'il y ait les mêmes données que celles utilisées pour l'entrainement
    donnees_test = pd.DataFrame({
        "bikes": bikes_zero,
        "annee": commit_at_index.year,
        "mois_sin": np.sin(2 * np.pi * commit_at_index.month / 12),
        "mois_cos": np.cos(2 * np.pi * commit_at_index.month / 12),
        "jour_sin": np.sin(2 * np.pi * commit_at_index.dayofweek / 7),
        "jour_cos": np.cos(2 * np.pi * commit_at_index.dayofweek / 7),
        "weekend": (commit_at_index.dayofweek >= 5).astype(int),
        "heure_sin": np.sin(2 * np.pi * commit_at_index.hour / 24),
        "heure_cos": np.cos(2 * np.pi * commit_at_index.hour / 24),
        "minute_sin": np.sin(2 * np.pi * commit_at_index.minute / 60),
        "minute_cos": np.cos(2 * np.pi * commit_at_index.minute / 60),
        "longitude": stations_pos["longitude"].values,
        "latitude": stations_pos["latitude"].values,
        "total": bikes_zero + stands_zero,
        "occupation": bikes_zero / (bikes_zero + stands_zero).replace(0, 1),
        "vacances": pd.Series(commit_at_index.date, index=test_modele.index).map(vacances_par_date).astype(int),
        "ferie": pd.Series(commit_at_index.date, index=test_modele.index).isin(tous_feries).astype(int),
    }, index=test_modele.index)
    
    
    
    # Valeurs déja connues dans le train -----------------------------------------------------------------------
    # On convertit le dictionnaire nom_vers_ancien en serie pandas
    station_fusion = pd.Index(station_index).map(lambda s: nom_vers_ancien.get(s, s))
    bikes_connus = train_modele["bikes"]
    
    
    
    # Prédictions ----------------------------------------------------------------------------------------------
    for col, h in decalage.items():
        print(h/1.2, "%")
        # On crée un vecteur avec l'horaire à prédire
        commit_at_cible = train["commit_at"] + pd.Timedelta(minutes=h)
        # On créé un dataset avec le nombre de vélos en prenant en compte le décalage horaire
        futur = train[["station", "commit_at", "bikes"]].rename(
            columns={"commit_at": "commit_at_cible", "bikes": "bikes_futur"}
        )
        # On remplit la coolonne
        avec_futur = train.assign(commit_at_cible=commit_at_cible).merge(
            futur, on=["station", "commit_at_cible"], how="inner"
        )
        
        # Prédictions
        model = LinearRegression()
        # On renseigne les données d'entrainement et leur réponse
        X = avec_futur[colonnes_train]
        y = avec_futur["bikes_futur"]
        # On entraine le modèle
        model.fit(X, y)
        # On prédit
        pred = model.predict(donnees_test[colonnes_train])
        # On empêche les prédictions d'être négatives ou de prédire plus de vélos que le nombre de places totales
        pred = np.clip(pred, 0, donnees_test["total"].values).round().astype(int)
        
        # Si la date demandée est dans train, on récupère directement la bonne valeur
        cibles = pd.MultiIndex.from_arrays([station_fusion, commit_at_index + pd.Timedelta(minutes=h)], names=["station", "commit_at"])
        # Création d'une série pandas avec le nombre de vélos à la date demandée
        connus = bikes_connus.reindex(cibles).to_numpy(dtype=float)
        # Quand connus existe, on l'utilise, et sinon on utilise la prédiction de la régression
        pred = np.where(np.isnan(connus), pred, connus).astype(int)
        print(f"{int((~np.isnan(connus)).sum())} valeurs lues dans train")
        
        # On écrit les prédictions dans test_modele
        test_modele[col] = pred
        
        # On libère la mémoire avant l'itération suivante
        del commit_at_cible, futur, avec_futur, X, y, model, cibles, connus
        gc.collect()
    
    # On renvoie les prédictions
    return test_modele

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
On entraine sur l'ensemble de validation et on fait une prédiction sur les dernières valeurs du train pour calculer le score RMSE perso <br>
Puis on ré-entraine sur TOUTES les données fournies pour faires les vraies prédictions <br>
Enfin on sauvegarde le résultat dans un fichier parquet
</div>

In [6]:
rmse, history_toulouse_prediction = RMSE_et_predictions(modele_prevision, "BA_prediction_104_RL")
print(f"RMSE : {rmse:.4f}")
history_toulouse_prediction

Nombre de stations dans train : 797
Nombre de stations UNIQUES dans train : 649
Nombre de mesures de base dans train : 40006230
Nombre de mesures après interpolation dans train : 40008580
12.5 %
132 valeurs lues dans train
25.0 %
132 valeurs lues dans train
37.5 %
132 valeurs lues dans train
50.0 %
132 valeurs lues dans train
62.5 %
132 valeurs lues dans train
75.0 %
132 valeurs lues dans train
87.5 %
132 valeurs lues dans train
100.0 %
132 valeurs lues dans train
Nombre de stations dans train : 800
Nombre de stations UNIQUES dans train : 650
Nombre de mesures de base dans train : 40012622
Nombre de mesures après interpolation dans train : 40013977
12.5 %
136 valeurs lues dans train
25.0 %
136 valeurs lues dans train
37.5 %
136 valeurs lues dans train
50.0 %
136 valeurs lues dans train
62.5 %
136 valeurs lues dans train
75.0 %
136 valeurs lues dans train
87.5 %
136 valeurs lues dans train
100.0 %
136 valeurs lues dans train
RMSE : 2.1756


,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,9,9,9,9,9,9,9,9
00002 - LAFAYETTE,2026-07-31 23:45:00,9,9,9,9,9,9,9,9
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,18,18,17,17,17,17,17,17
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,12,12,12,11,11,11,11,11
00003 - POMME,2025-01-05 05:45:00,17,17,16,16,16,16,15,15
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0,0,0,1,1,1,1,1
PASTEUR - LAVOISIER,2025-08-05 10:15:00,0,0,0,0,0,0,0,0
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,0,0,0,0,0,0,0,0
